In [ ]:
# ============================================================
# CANONICAL OPENCLIP EVALUATION
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPModel, CLIPProcessor

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT = (
    "/content/drive/MyDrive/"
    "Surgical-VLM/checkpoints/best_openclip"
)

RESULTS_DIR = Path(
    "/content/drive/MyDrive/"
    "Surgical-VLM/results"
)

print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT)
print("Results directory:", RESULTS_DIR)

Device: cpu
Checkpoint: /content/drive/MyDrive/Surgical-VLM/checkpoints/best_openclip
Results directory: /content/drive/MyDrive/Surgical-VLM/results


In [ ]:
# ============================================================
# LOAD MODEL
# ============================================================

processor = CLIPProcessor.from_pretrained(
    CHECKPOINT
)

model = CLIPModel.from_pretrained(
    CHECKPOINT
)

model = model.to(DEVICE)
model.eval()

print("OpenCLIP loaded successfully.")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

OpenCLIP loaded successfully.


In [ ]:
# ============================================================
# LOAD EVALUATION DATA
# ============================================================

EVAL_PATH = (
    RESULTS_DIR /
    "OpenCLIP Results/evaluation_df.parquet"
)

evaluation_df = pd.read_parquet(
    EVAL_PATH
)

print(
    "Evaluation shape:",
    evaluation_df.shape
)

print(
    "\nColumns:"
)

print(
    evaluation_df.columns.tolist()
)

display(
    evaluation_df.head()
)

Evaluation shape: (5600, 18)

Columns:
['sample_id', 'image_id', 'case_id', 'image_path', 'image_full_path', 'image_exists', 'source_dataset', 'specialty', 'surgery_type', 'question', 'thinking', 'answer', 'task', 'task_group', 'gt_label', 'annotator', 'text', 'label_set']


,sample_id,image_id,case_id,image_path,image_full_path,image_exists,source_dataset,specialty,surgery_type,question,thinking,answer,task,task_group,gt_label,annotator,text,label_set
0,517834,29615,VID110,CholecT50/videos/VID110/001493.png,/content/CholecT50/CholecT50/videos/VID110/001...,True,CholecT50,Hepatobiliary,Cholecystectomy,Identify the grasper's action in this surgery ...,None,The grasper is performing a retract action in ...,Action Recognition,Understanding and Reasoning,"[{""actions"": [""retract""]}]",NUS,The grasper is performing a retract action in ...,[]
1,32083,67707,VID68,CholecT50/videos/VID68/001508.png,/content/CholecT50/CholecT50/videos/VID68/0015...,True,CholecT50,Hepatobiliary,Cholecystectomy,"Given the laparoscopic cholecystectomy image, ...",This is a Level 2 task requiring identificatio...,"retract, dissect",Action Recognition,Understanding and Reasoning,"[""retract, dissect""]",SJTU,"retract, dissect",[]
2,552118,31976,VID36,CholecT50/videos/VID36/000224.png,/content/CholecT50/CholecT50/videos/VID36/0002...,True,CholecT50,Hepatobiliary,Cholecystectomy,What is the grasper doing in this surgical scene?,None,The grasper is performing a retract action in ...,Action Recognition,Understanding and Reasoning,"[{""actions"": [""retract""]}]",NUS,The grasper is performing a retract action in ...,[]
3,33687,29387,VID05,CholecT50/videos/VID05/000293.png,/content/CholecT50/CholecT50/videos/VID05/0002...,True,CholecT50,Hepatobiliary,Cholecystectomy,"Given the laparoscopic cholecystectomy image, ...",This is a Level 2 task requiring identificatio...,"retract, dissect",Action Recognition,Understanding and Reasoning,"[""retract, dissect""]",SJTU,"retract, dissect",[]
4,41279,31695,VID36,CholecT50/videos/VID36/001288.png,/content/CholecT50/CholecT50/videos/VID36/0012...,True,CholecT50,Hepatobiliary,Cholecystectomy,"Given the laparoscopic cholecystectomy image, ...",This is a Level 2 task requiring identificatio...,dissect,Action Recognition,Understanding and Reasoning,"[""dissect""]",SJTU,dissect,[]


In [ ]:
# ============================================================
# LOAD IMAGE EMBEDDINGS
# ============================================================

IMAGE_EMBEDDINGS_PATH = (
    RESULTS_DIR /
    "OpenCLIP Results/image_embeddings.npy"
)

image_embeddings = np.load(
    IMAGE_EMBEDDINGS_PATH
)

print(
    "Image embedding shape:",
    image_embeddings.shape
)

Image embedding shape: (5600, 512)


In [ ]:
# ============================================================
# CANONICAL TASK-SPECIFIC LABEL POOLS
# ============================================================

CANONICAL_POOLS = {

    "Instrument Recognition": [
        "grasper",
        "bipolar",
        "hook",
        "scissors",
        "clipper",
        "irrigator",
    ],

    "Action Recognition": [
        "grasp",
        "retract",
        "dissect",
        "coagulate",
        "clip",
        "cut",
        "aspirate",
        "irrigate",
        "pack",
        "null_verb",
    ],

    "Tissue and Organ Recognition": [
        "gallbladder",
        "cystic artery",
        "cystic duct",
        "cystic plate",
        "liver",
        "specimen bag",
        "fluid",
        "abdominal wall cavity",
        "omentum",
        "blood_vessel",
        "gut",
        "peritoneum",
        "cystic_pedicle",
        "adhesion",
        "null_target",
        # Include the remaining canonical tissue labels
        # used in your VLM evaluation here.
    ],

    "Phase Recognition": [
        "Preparation",
        "Calot Triangle Dissection",
        "Clipping Cutting",
        "Gallbladder Dissection",
        "Gallbladder Packaging",
        "Cleaning Coagulation",
        "Gallbladder Retraction",
    ],
}

In [ ]:
print(
    len(CANONICAL_POOLS["Tissue and Organ Recognition"])
)

print(
    CANONICAL_POOLS["Tissue and Organ Recognition"]
)

15
['gallbladder', 'cystic artery', 'cystic duct', 'cystic plate', 'liver', 'specimen bag', 'fluid', 'abdominal wall cavity', 'omentum', 'blood_vessel', 'gut', 'peritoneum', 'cystic_pedicle', 'adhesion', 'null_target']


In [ ]:
# ============================================================
# LABEL NORMALIZATION
# ============================================================

def normalize_label(label):

    label = str(label).strip().lower()

    label = label.replace("_", " ")

    label = " ".join(
        label.split()
    )

    return label

In [ ]:
# ============================================================
# GROUND-TRUTH LABEL MATCHING
# ============================================================

def extract_canonical_labels(
    text,
    candidate_labels
):

    text = normalize_label(text)

    found = []

    # Longest labels first prevents shorter labels
    # from interfering with multi-word labels.
    sorted_labels = sorted(
        candidate_labels,
        key=len,
        reverse=True
    )

    for label in sorted_labels:

        normalized = normalize_label(
            label
        )

        if normalized in text:
            found.append(label)

    return found

In [ ]:
# ============================================================
# GROUND-TRUTH AUDIT
# ============================================================

for task, labels in CANONICAL_POOLS.items():

    print("\n" + "=" * 70)
    print(task)
    print("=" * 70)

    subset = evaluation_df[
        evaluation_df["task"] == task
    ]

    for _, row in subset.head(10).iterrows():

        gt = extract_canonical_labels(
            row["answer"],
            labels
        )

        print(
            "\nRaw:",
            row["answer"]
        )

        print(
            "Extracted:",
            gt
        )


Instrument Recognition

Raw: hook
Extracted: ['hook']

Raw: bipolar
Extracted: ['bipolar']

Raw: grasper
Extracted: ['grasper']

Raw: ## Surgical Answers
(1) They fit into Clipper.
(2) The identified phase is Clipping Cutting.
Extracted: ['clipper']

Raw: ## Surgical Answers
(1) They fall under the categories Grasper, Hook.
(2) The identified phase is Gallbladder Dissection.
Extracted: ['grasper', 'hook']

Raw: grasper, hook
Extracted: ['grasper', 'hook']

Raw: hook
Extracted: ['hook']

Raw: They are classified into the categories: grasper, hook.
Extracted: ['grasper', 'hook']

Raw: hook
Extracted: ['hook']

Raw: hook
Extracted: ['hook']

Action Recognition

Raw: The grasper is performing a retract action in this surgery image.
Extracted: ['retract', 'grasp']

Raw: retract, dissect
Extracted: ['retract', 'dissect']

Raw: The grasper is performing a retract action in this surgical scene.
Extracted: ['retract', 'grasp']

Raw: retract, dissect
Extracted: ['retract', 'dissect']

Raw: diss

In [ ]:
# ============================================================
# CANONICAL LABEL EMBEDDINGS
# ============================================================

canonical_embeddings = {}

with torch.no_grad():

    for task, labels in CANONICAL_POOLS.items():

        inputs = processor(
            text=labels,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }

        text_outputs = model.text_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

        text_features = model.text_projection(
            text_outputs.pooler_output
        )

        text_features = torch.nn.functional.normalize(
            text_features,
            dim=-1
        )

        canonical_embeddings[task] = (
            text_features
            .cpu()
            .numpy()
        )

        print(
            task,
            "→",
            text_features.shape
        )

Instrument Recognition → torch.Size([6, 512])
Action Recognition → torch.Size([10, 512])
Tissue and Organ Recognition → torch.Size([15, 512])
Phase Recognition → torch.Size([7, 512])


In [ ]:
# ============================================================
# NORMALIZE IMAGE EMBEDDINGS
# ============================================================

image_embeddings = (
    image_embeddings /
    np.linalg.norm(
        image_embeddings,
        axis=1,
        keepdims=True
    )
)

print(
    "Image embeddings normalized."
)

Image embeddings normalized.


In [ ]:
# ============================================================
# CANONICAL OPENCLIP RETRIEVAL
# ============================================================

results = []

for idx, row in tqdm(
    evaluation_df.iterrows(),
    total=len(evaluation_df)
):

    task = row["task"]

    if task not in CANONICAL_POOLS:
        continue

    labels = CANONICAL_POOLS[task]

    text_embeds = canonical_embeddings[task]

    image_embed = image_embeddings[idx]

    similarities = (
        text_embeds @ image_embed
    )

    best_idx = np.argmax(
        similarities
    )

    predicted_label = labels[
        best_idx
    ]

    results.append({

        "original_index": idx,

        "task": task,

        "ground_truth_text":
            row["answer"],

        "predicted_label":
            predicted_label,

        "similarity":
            similarities[best_idx],

        "candidate_count":
            len(labels),
    })

canonical_results = pd.DataFrame(
    results
)

print(
    "Results shape:",
    canonical_results.shape
)

display(
    canonical_results.head()
)

  0%|          | 0/5600 [00:00<?, ?it/s]

Results shape: (3200, 6)


,original_index,task,ground_truth_text,predicted_label,similarity,candidate_count
0,0,Action Recognition,The grasper is performing a retract action in ...,grasp,0.203857,10
1,1,Action Recognition,"retract, dissect",dissect,0.220388,10
2,2,Action Recognition,The grasper is performing a retract action in ...,retract,0.215450,10
3,3,Action Recognition,"retract, dissect",retract,0.193866,10
4,4,Action Recognition,dissect,dissect,0.233671,10


In [ ]:
# ============================================================
# TOP-1 CORRECTNESS
# ============================================================

canonical_results["ground_truth_labels"] = (
    canonical_results.apply(
        lambda row:
        extract_canonical_labels(
            row["ground_truth_text"],
            CANONICAL_POOLS[row["task"]]
        ),
        axis=1
    )
)

canonical_results["top1_correct"] = (
    canonical_results.apply(
        lambda row:
        row["predicted_label"]
        in row["ground_truth_labels"],
        axis=1
    )
)

display(
    canonical_results.head()
)

,original_index,task,ground_truth_text,predicted_label,similarity,candidate_count,ground_truth_labels,top1_correct
0,0,Action Recognition,The grasper is performing a retract action in ...,grasp,0.203857,10,"[retract, grasp]",True
1,1,Action Recognition,"retract, dissect",dissect,0.220388,10,"[retract, dissect]",True
2,2,Action Recognition,The grasper is performing a retract action in ...,retract,0.215450,10,"[retract, grasp]",True
3,3,Action Recognition,"retract, dissect",retract,0.193866,10,"[retract, dissect]",True
4,4,Action Recognition,dissect,dissect,0.233671,10,[dissect],True


In [ ]:
# ============================================================
# CANONICAL OPENCLIP RESULTS
# ============================================================

summary_rows = []

for task in CANONICAL_POOLS:

    subset = canonical_results[
        canonical_results["task"] == task
    ]

    accuracy = (
        subset["top1_correct"].mean()
    )

    candidate_count = (
        len(CANONICAL_POOLS[task])
    )

    chance = (
        1.0 /
        candidate_count
    )

    lift = (
        accuracy - chance
    )

    summary_rows.append({

        "Task": task,

        "Candidate count":
            candidate_count,

        "N evaluated":
            len(subset),

        "Top-1 accuracy":
            accuracy,

        "Chance":
            chance,

        "Lift over chance":
            lift,
    })

canonical_summary = pd.DataFrame(
    summary_rows
)

display(
    canonical_summary
)

,Task,Candidate count,N evaluated,Top-1 accuracy,Chance,Lift over chance
0,Instrument Recognition,6,800,0.90875,0.166667,0.742083
1,Action Recognition,10,800,0.60125,0.100000,0.501250
2,Tissue and Organ Recognition,15,800,0.69250,0.066667,0.625833
3,Phase Recognition,7,800,0.76625,0.142857,0.623393


In [ ]:
MAJORITY_LABELS = {

    "Instrument Recognition":
        "hook",

    "Action Recognition":
        "dissect",

    "Tissue and Organ Recognition":
        "gallbladder",

    "Phase Recognition":
        "Gallbladder Dissection",
}

In [ ]:
# ============================================================
# MAJORITY BASELINE
# ============================================================

majority_rows = []

for task, majority_label in MAJORITY_LABELS.items():

    subset = canonical_results[
        canonical_results["task"] == task
    ]

    correct = (
        subset["ground_truth_labels"]
        .apply(
            lambda labels:
            majority_label in labels
        )
    )

    majority_accuracy = (
        correct.mean()
    )

    majority_rows.append({

        "Task": task,

        "Majority label":
            majority_label,

        "Majority baseline":
            majority_accuracy,

        "N":
            len(subset),
    })

majority_summary = pd.DataFrame(
    majority_rows
)

display(
    majority_summary
)

,Task,Majority label,Majority baseline,N
0,Instrument Recognition,hook,0.69625,800
1,Action Recognition,dissect,0.50875,800
2,Tissue and Organ Recognition,gallbladder,0.82125,800
3,Phase Recognition,Gallbladder Dissection,0.39875,800


In [ ]:
# ============================================================
# FINAL CANONICAL OPENCLIP TABLE
# ============================================================

final_openclip_table = (
    canonical_summary
    .merge(
        majority_summary,
        on="Task",
        how="left"
    )
)

display(
    final_openclip_table
)

,Task,Candidate count,N evaluated,Top-1 accuracy,Chance,Lift over chance,Majority label,Majority baseline,N
0,Instrument Recognition,6,800,0.90875,0.166667,0.742083,hook,0.69625,800
1,Action Recognition,10,800,0.60125,0.100000,0.501250,dissect,0.50875,800
2,Tissue and Organ Recognition,15,800,0.69250,0.066667,0.625833,gallbladder,0.82125,800
3,Phase Recognition,7,800,0.76625,0.142857,0.623393,Gallbladder Dissection,0.39875,800


In [ ]:
# ============================================================
# VLM COMPARISON
# ============================================================

vlm_results = pd.DataFrame({

    "Task": [
        "Instrument Recognition",
        "Action Recognition",
        "Tissue and Organ Recognition",
    ],

    "VLM zero-shot F1": [
        0.055,
        0.399,
        0.569,
    ],

    "VLM fine-tuned F1": [
        0.613,
        0.608,
        0.520,
    ],
})

display(vlm_results)

,Task,VLM zero-shot F1,VLM fine-tuned F1
0,Instrument Recognition,0.055,0.613
1,Action Recognition,0.399,0.608
2,Tissue and Organ Recognition,0.569,0.520


In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

canonical_results.to_parquet(
    RESULTS_DIR /
    "openclip_canonical_results.parquet",
    index=False
)

final_openclip_table.to_csv(
    RESULTS_DIR /
    "openclip_canonical_summary.csv",
    index=False
)

print(
    "Canonical OpenCLIP results saved."
)

Canonical OpenCLIP results saved.
